In [25]:
import os
from getpass import getpass

from dotenv import load_dotenv



In [26]:

load_dotenv()

def set_api_key_if_not_present(key_name, prompt_message=""):
    if len(prompt_message) == 0:
        prompt_message=key_name
    if key_name not in os.environ or not os.environ[key_name]:
        os.environ[key_name] = getpass.getpass(prompt_message)

set_api_key_if_not_present("OPENAI_API_KEY")

# Data Preparation

First, we will read in the transcripts of the videos and convert them to Documents
with appropriate metadata.

In [27]:
filename = "../test.json"


In [37]:
import json

data = json.load(open(filename, "rb"))
data[0]

{'video_id': 4157,
 'title': 'Get organized with layer groups',
 'desc': 'Learn some great tips for working with layers.',
 'length': '00:04:05.78',
 'url': 'https://videos-tv.adobe.com/2013-07-23/f65b5a0ef188ba5e5a96df93a8ead3cf.mp4',
 'transcripts': [{'sent_id': 0,
   'sent': "At any time when you're working in Adobe Photoshop on a complicated subject the possibility exists of chaos ensuing.",
   'begin': 1.27,
   'end': 9.97},
  {'sent_id': 1,
   'sent': "Well, you know what chaos there is, don't you?",
   'begin': 10.16,
   'end': 12.01},
  {'sent_id': 2,
   'sent': "It's my studio in the middle of a project.",
   'begin': 12.009999,
   'end': 14.23},
  {'sent_id': 3,
   'sent': 'What I would like to do is reduce the clutter, reduce the chaos.',
   'begin': 14.58,
   'end': 17.719999},
  {'sent_id': 4,
   'sent': 'Let me move Layers over here again, I would say that is not really necessary, but it makes life easier.',
   'begin': 17.719999,
   'end': 23.15},
  {'sent_id': 5,
   'se

In [38]:
from langchain_core.document_loaders import BaseLoader
from typing import List, Dict, Iterator
from langchain_core.documents import Document

class VideoTranscriptBulkLoader(BaseLoader):
    """Loads video transcripts as a bulk into documents"""

    def __init__(self, json_payload:List[Dict]):

        self.json_payload = json_payload
        
    def lazy_load(self) -> Iterator[Document]:
        """Lazy loader that returns an iterator"""
        
        for video in self.json_payload:
            metadata = dict(video)
            metadata.pop("transcripts", None)
            metadata.pop("qa", None)
            # Rename 'url' key to 'source' in metadata if it exists
            if "url" in metadata:
                metadata["source"] = metadata.pop("url")
            yield Document(
                page_content = "\n".join(t["sent"] for t in video["transcripts"]),
                metadata = metadata
            )


loader = VideoTranscriptBulkLoader(data)
docs = loader.load()



## R - retrieval

Let's hit it with a semantic chunker.

In [47]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [49]:
text_splitter = SemanticChunker(embeddings)

split_documents = text_splitter.split_documents(docs)


In [55]:
pprint( split_documents[1].page_content )

("It doesn't really hurt a thing, gives me total control. But I would like to "
 'reduce the clutter in my Layers panel by creating something called a Group. '
 "So let's do this a couple of ways. Number one, we can come down here and "
 'click this button right there. That\'s a "Group" button. Go and click it. It '
 "will give you a group called: Group 1. Now, there is nothing in it yet, it's "
 'just there. To get these elements in here is actually pretty easy. Select '
 'this one and "Shift" click down here - assuming you want them all. Now '
 "actually maybe you don't. Let me show you something else here. If you click "
 'on this one and hold down the "Command" key on a Macintosh or the "Control" '
 'key in Windows you can select, well, every other one if you want to. If you '
 'want to select them all - "Shift" click. Now I\'ll just drag them up to '
 'Group 1 Until you see a little box kind of come around Group 1 and let go. '
 'They should indent if it worked.')


In [48]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

collection_name = f"{filename}_qdrant"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
)

/home/mbudisic/Documents/PsTuts-VQA-Dataset/.venv/lib/python3.12/site-packages/qdrant_client/http/models/models.py:758: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is empty, alternative syntax for `is_empty: \&quot;field_name\&quot;`",
/home/mbudisic/Documents/PsTuts-VQA-Dataset/.venv/lib/python3.12/site-packages/qdrant_client/http/models/models.py:762: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is null, alternative syntax for `is_null: \&quot;field_name\&quot;`",


In [56]:
_ = vector_store.add_documents(documents=split_documents)

In [57]:
retriever = vector_store.as_retriever(search_kwargs={"k":2})

def retrieve(state):
    retrieved_docs = retriever.invoke(state["question"])
    return {"context":retrieved_docs}


In [63]:
a = retrieve({"question":"What is a layer?"})
[ pprint(d.page_content) for d in a["context"] ]

("Layers are the building blocks of any image in Photoshop CC. So, it's "
 "important to understand, what layers are and why to use them - which we'll "
 "cover in this video. If you're following along, open this layered image from "
 'the downloadable practice files for this tutorial. You might think of layers '
 'like separate flat pints of glass, stacked one on top of the other. Each '
 'layer contains separate pieces of content. To get a sense of how layers are '
 "constructed, let's take a look at this Layers panel. I've closed my other "
 'panels, so that we can focus on the Layers panel. But you can skip that. By '
 "the way: If your Layers panel isn't showing, go up to the Window menu and "
 'choose Layers from there. The Layers panel is where you go to select and '
 'work with layers. In this image there are 4 layers, each with separate '
 'content. If you click the Eye icon to the left of a layer, you can toggle '
 "the visibility of that layer off and on. So, I'm going to tu

[None, None]

## A - Augmentation

We need to populate a prompt for LLM.


In [191]:
docs[0].metadata

{'video_id': 4157,
 'title': 'Get organized with layer groups',
 'desc': 'Learn some great tips for working with layers.',
 'length': '00:04:05.78',
 'source': 'https://videos-tv.adobe.com/2013-07-23/f65b5a0ef188ba5e5a96df93a8ead3cf.mp4'}

In [192]:
from langchain.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. 
Your answers use emojis for emphasis.

IMPORTANT: You must only use the provided context, and cannot use your own knowledge.

If there is no context that corresponds to the query, respond by saying
"I don't know. This is not available in our training library."

In particular, you are an expert on Photoshop and your goal is to help users
gain knowledge from a database of training videos.

Most of the users questions will be in the form:
"How can I do ..."
or
"What is ..."

When appropriate, provide your answers in a step-by-step form.
ALWAYS list the URL and the title of the reference video.
NEVER invent the explanation. ALWAYS use ONLY the context information.

"""

RAG_PROMPT="""\

### Question
{question}

ALWAYS list the "url" field from the context.
NEVER invent the explanation. ALWAYS use ONLY the context information.

### Context
{context}

### References
{references}

"""

rag_prompt = ChatPromptTemplate(
    [("system",SYSTEM_PROMPT), 
     ("human",RAG_PROMPT)
     ]
    )

## Generation

We will use a 4.1-nano to generate answers.

In [193]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano",temperature=0)

In [194]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  docs_urls = "\n\n".join(doc.metadata["source"] for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], 
                                        context=docs_content,
                                        references=docs_urls)
  response = llm.invoke(messages)
  return {"response" : response.content}

In [195]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str
  

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [199]:
response = graph.invoke({"question" : "What is the best way of changing the AI model of your file?"})

In [200]:
response

{'question': 'What is the best way of changing the AI model of your file?',
 'context': [Document(metadata={'video_id': 19214, 'title': 'Replace a background using a layer mask', 'desc': 'Use a layer mask to replace one background with another.\xa0', 'length': '00:05:06.11', 'source': 'https://images-tv.adobe.com/avp/vr/b758b4c4-2a74-41f4-8e67-e2f2eab83c6a/a9e61350-7849-4a24-8b11-361058f835e3/c16f8d91-4830-461f-be76-398e6801c9e6_20170727094418.1280x720at2400_h264.mp4', '_id': 'a571330d71484cdea3b329c7c92843e6', '_collection_name': '../test.json_qdrant'}, page_content="And we'll learn more about layer masking in the process. We'll start with this photo from the practice files of an art piece by a wood artist. Let's replace its background with a more interesting shot from the artist studio. The first step is to bring in another background image. To do that go up to the File menu, choose Place Embedded..., and navigate to this image and click Place. Go to the Options bar and click the Che

In [201]:
pprint(response["response"])

"I don't know. This is not available in our training library."
